# Filamentation SiO2 — 1030 nm, 4 µJ

| | |
|---|---|
| sonde | 515 nm |
| énergie | 4 µJ incidents (frame expérimentale du 20260804) |
| profil 1/e² | sx/sy = 11.5 / 11.0 µm → rayon équivalent 11.25 µm |
| waist | z = 0 |
| boîte | 0 → 350 µm |
| tracés | r = (−70, 70) µm, z = (0, 350) µm, OPL (nm) et transmittance |

Exécuter les cellules dans l'ordre.


## Installation des dépendances

À exécuter en premier. Le post-traitement ne demande que numpy / scipy / matplotlib ; `cupy` n'est nécessaire que pour lancer un nouveau calcul sur GPU.


In [ ]:
# [deps-install-cell]
# ============================================================================
#  Installation des dependances
# ============================================================================
# A executer une fois, en premier. Si cupy est installe par cette cellule,
# REDEMARRER LE NOYAU avant de continuer.
#
# Deux niveaux :
#   - CPU  : numpy / scipy / matplotlib (+ pillow). Suffisent pour tout
#            le POST-TRAITEMENT, c'est-a-dire relire un result.npz deja calcule
#            et regenerer les figures.
#   - GPU  : cupy. Necessaire uniquement pour LANCER un calcul
#            (filament_sim.run()), qui est un solveur CUDA.
import importlib
import importlib.util
import re
import shutil
import subprocess
import sys

REQUIRED = ["numpy", "scipy", "matplotlib", "pillow"]
_IMPORT_NAME = {"pillow": "PIL", "ipywidgets": "ipywidgets"}


def _pip(*args):
    print("  pip install", *args)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


print("Dependances CPU :")
for pkg in REQUIRED:
    mod = _IMPORT_NAME.get(pkg, pkg)
    if importlib.util.find_spec(mod) is None:
        _pip(pkg)
    else:
        print(f"  {pkg:12s} deja present")

print("\nDependance GPU (cupy) :")
if importlib.util.find_spec("cupy") is not None:
    import cupy
    print(f"  cupy {cupy.__version__} deja present")
    try:
        print(f"  GPU visible : {cupy.cuda.runtime.getDeviceCount()} device(s)")
    except Exception as exc:
        print(f"  /!\\ cupy importe mais aucun GPU utilisable ({type(exc).__name__})")
else:
    # La roue cupy depend de la version de CUDA du pilote : il n'existe pas de
    # paquet "cupy" generique qui marche partout, d'ou la detection.
    cuda_major = None
    if shutil.which("nvidia-smi"):
        try:
            out = subprocess.check_output(["nvidia-smi"], text=True)
            m = re.search(r"CUDA Version:\s*(\d+)\.", out)
            cuda_major = int(m.group(1)) if m else None
        except Exception:
            pass
    if cuda_major is None:
        print("  aucun GPU NVIDIA detecte -> cupy N'EST PAS installe.")
        print("  Le post-traitement fonctionne quand meme sur un result.npz")
        print("  deja calcule ; seul filament_sim.run() a besoin du GPU.")
    else:
        _pip(f"cupy-cuda{12 if cuda_major >= 12 else 11}x")
        print("  cupy installe -> REDEMARRER LE NOYAU avant de continuer.")

print("\nVersions :")
for _m in ("numpy", "scipy", "matplotlib"):
    try:
        print(f"  {_m:12s} {importlib.import_module(_m).__version__}")
    except ImportError:
        print(f"  {_m:12s} ABSENT")


## 0. Imports

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.constants import c as c_SI, epsilon_0, m_e, elementary_charge as q_e

for _p in (Path.cwd().parent / "sim", Path.cwd() / "sim"):
    sys.path.insert(0, str(_p))

from filament_sim import run, FIELD_TOGGLES, n_sellmeier
import figures_filament as ff
import figures_report as fr

OUT_ROOT = Path("runs_z0")
FIG_DIR = OUT_ROOT / "figures"
REP_DIR = OUT_ROOT / "figures_rapport"
OUT_ROOT.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)
REP_DIR.mkdir(exist_ok=True)
print("toggles :", FIELD_TOGGLES)

## 1. Paramètres

In [ ]:
# ---------------- laser ----------------
WAVELENGTH_M = 1030e-9
SX_UM        = 11.5
SY_UM        = 11.0
W0_M         = np.sqrt(SX_UM * SY_UM) * 1e-6
DELTA_T_S    = 263e-15

FRESNEL_T           = 1.0 - ((1.45 - 1.0) / (1.45 + 1.0))**2
ENERGY_INCIDENT_UJ  = 4.0
ENERGY_SWEEP_INC_UJ = [4.0]

# ---------------- materiau ----------------
N2          = 2.74e-20
UI_EV       = 9.0
MEFF_REL    = 0.64   # masse reduite du taux de Keldysh
MEFF_DRUDE_REL = 1.0 # masse effective du terme de Drude (sigma, avalanche)
TAU_C_S     = 1.7e-15
TAU_R_S     = 330e-15
TAU_STE_S   = 1e-12
RHO_MAX_CM3 = 2.1e22
US_EV       = 6.0
E_TR_EV     = 4.2
F_R         = 0.18
TAU_D_S     = 32e-15
TAU_S_S     = 12e-15

# ---------------- sonde ----------------
LAMBDA_PROBE_M = 515e-9

# ---------------- boite ----------------
BEGIN_M = 0.0
END_M   = 350e-6

# ---------------- grille ----------------
FAST = dict(Nz=9000,  Nt=2048, Nr=1024, R_factor=8.0,
            save_stride=20, rho_t_stride=16, rho_r_stride=2)
PROD = dict(Nz=18000, Nt=4096, Nr=2048, R_factor=8.0,
            save_stride=40, rho_t_stride=16, rho_r_stride=4)
GRID = FAST

# ---------------- trace ----------------
Z_LIM_EXP = (0.0, 350.0)
R_LIM_EXP = (-70.0, 70.0)
X_HALF_UM = 75.0
DELAYS_FS = [-500, 0, 250, 500, 1000, 1500, 1901, 2000]  # 1901 fs = la frame experimentale

print("parametres definis")

## 2. Grandeurs dérivées

In [ ]:
ENERGY_IN_GLASS_UJ = ENERGY_INCIDENT_UJ * FRESNEL_T
ENERGY_SWEEP_UJ    = [E * FRESNEL_T for E in ENERGY_SWEEP_INC_UJ]

n0 = n_sellmeier(WAVELENGTH_M)
k0 = 2.0 * np.pi * n0 / WAVELENGTH_M
zR = k0 * W0_M**2 / 2.0

tp   = DELTA_T_S / np.sqrt(2.0 * np.log(2.0))
tmax = 5.0 * tp
dt   = 2.0 * tmax / GRID["Nt"]
dz   = (END_M - BEGIN_M) / GRID["Nz"]
R_MAX = GRID["R_factor"] * W0_M
dr   = R_MAX / (GRID["Nr"] - 1)

n_saves = GRID["Nz"] // GRID["save_stride"] + 1
Nt_sub  = (GRID["Nt"] - 1) // GRID["rho_t_stride"] + 1
Nr_sub  = (GRID["Nr"] - 2) // GRID["rho_r_stride"] + 1

NC_PROBE = epsilon_0 * m_e * (2.0*np.pi*c_SI/LAMBDA_PROBE_M)**2 / q_e**2 * 1e-6

PROBE_KW = dict(lambda_probe_m=LAMBDA_PROBE_M, E_tr_eV=E_TR_EV, n2=N2,
                tau_c_s=TAU_C_S, tau_r_s=TAU_R_S, tau_ste_s=TAU_STE_S)

print(f"w0        = {W0_M*1e6:.2f} um       z_R = {zR*1e6:.0f} um")
print(f"sonde     = {LAMBDA_PROBE_M*1e9:.0f} nm     n_c = {NC_PROBE:.3e} cm-3")
print(f"energie   = {ENERGY_INCIDENT_UJ} uJ incidents -> {ENERGY_IN_GLASS_UJ:.3f} uJ dans le verre")
print(f"balayage  = {ENERGY_SWEEP_INC_UJ} uJ incidents")
print(f"z : Nz={GRID['Nz']}  dz={dz*1e9:.1f} nm  |  {n_saves} plans, dz_save={dz*GRID['save_stride']*1e6:.2f} um")
print(f"r : R_max={R_MAX*1e6:.0f} um  dr={dr*1e9:.0f} nm  ({W0_M/dr:.0f} pts dans w0)")
print(f"t : Nt={GRID['Nt']}  dt={dt*1e15:.2f} fs  f_Nyq/f0={1.0/(2*dt)/(c_SI/WAVELENGTH_M):.2f}")
print(f"cube (z,r,t) : {3*n_saves*Nr_sub*Nt_sub*4/1e6:.0f} Mo")

## 3. Contrôles avant lancement

In [ ]:
P_cr = ff.critical_power(N2, WAVELENGTH_M, n0)
print(f"P_cr = {P_cr*1e-6:.2f} MW\n")

for E_inc, E_gl in zip(ENERGY_SWEEP_INC_UJ, ENERGY_SWEEP_UJ):
    P_in = E_gl * 1e-6 / (tp * np.sqrt(np.pi / 2.0))
    ratio, L_DF, L_c, _ = ff.marburger_collapse(P_in, P_cr, W0_M, WAVELENGTH_M, n0)
    print(f"--- {E_inc:g} uJ incidents ({E_gl:.2f} uJ verre) ---")
    ff.check_entrance_intensity(E_gl, W0_M, DELTA_T_S, BEGIN_M, WAVELENGTH_M, n0)
    print(f"  P/P_cr = {ratio:.1f}   L_c = {L_c*1e6:.0f} um\n")

## 4. Les trois runs

In [ ]:
sweep = {}

for E_inc, E_gl in zip(ENERGY_SWEEP_INC_UJ, ENERGY_SWEEP_UJ):
    tag = f"{E_inc:g}uJ"
    out_dir = str(OUT_ROOT / f"z0_350um_{tag}")
    result = ff.load_scenario_npz(out_dir)
    if result is None:
        print(f"\n=== lancement {tag} ===")
        result = run(
            Nz=GRID["Nz"], Nt=GRID["Nt"], Nr=GRID["Nr"], R_factor=GRID["R_factor"],
            begin=BEGIN_M, end=END_M,
            save_stride=GRID["save_stride"], ckpt_every=200, verbose=True,
            wavelength=WAVELENGTH_M, energy_uJ=E_gl,
            w0=W0_M, delta_t=DELTA_T_S,
            n2=N2, Ui_eV=UI_EV, meff_rel=MEFF_REL,
            tau_c=TAU_C_S, tau_r=TAU_R_S, rho_max=RHO_MAX_CM3,
            Us_eV=US_EV, tau_ste=TAU_STE_S,
            f_R=F_R, tau_d=TAU_D_S, tau_s=TAU_S_S,
            enable_ste=True, lambda_probe=LAMBDA_PROBE_M,
            rho_t_stride=GRID["rho_t_stride"], rho_r_stride=GRID["rho_r_stride"],
            out_dir=out_dir, envelope="gaussian_focused",
        )
    sweep[tag] = result
    ff.run_health_check(result, out_dir=out_dir, label=tag, rho_max=RHO_MAX_CM3)
    print()

res = sweep[f"{ENERGY_INCIDENT_UJ:g}uJ"]
print("runs disponibles :", list(sweep))

## 5. Diagnostics

In [ ]:
ff.plot_fig8_peak_intensity(sweep, save=str(FIG_DIR / "peak_intensity.png"))
ff.plot_fig9_electron_density(sweep, nc_probe_cm3=NC_PROBE,
                              save=str(FIG_DIR / "rho_e.png"))
ff.plot_fig7_fluence_contours(res, levels=(1.0, 5.0, 20.0, 50.0),
                              label=f"{ENERGY_INCIDENT_UJ:g} uJ",
                              save=str(FIG_DIR / "fluence.png"))
ff.plot_free_vs_trapped_vs_z(res, rho_max_cm3=RHO_MAX_CM3, nc_probe_cm3=NC_PROBE,
                             save=str(FIG_DIR / "rho_e_rho_s.png"))

for tag in sweep:
    print(tag)
    ff.count_refocusing_cycles(sweep[tag])

## 6. Planches OPL + transmittance

Format expérimental exact : `r = (−70, 70) µm`, `z = (0, 350) µm`.

In [ ]:
for tag in sweep:
    for delay in (0.0, 500.0, 1000.0):
        fig, diag = ff.plot_opl_panel(
            sweep[tag], delay_fs=delay,
            z_lim=Z_LIM_EXP, r_lim=R_LIM_EXP, x_half_um=X_HALF_UM,
            opl_clip_nm=15.0, t_lim=(0.75, 1.15),
            rho_max_cm3=RHO_MAX_CM3, validity_frac=0.1,
            title=f"simulation {tag}, delay {delay/1000:+.3f} ps",
            save=str(FIG_DIR / f"opl_{tag}_{delay:+05.0f}fs.png"),
            **PROBE_KW)
        print(f"{tag:6s} {delay:+6.0f} fs :"
              f"  OPL max = {np.abs(diag['opl_nm']).max():8.2f} nm"
              f"   T min = {diag['transmittance'].min():.3f}")
    print()

## 6b. Contrôles d'intégration

Les planches reposent sur deux intégrales numériques sans vérité de référence :
la transformée d'Abel le long de la corde (vue de côté) et l'intégrale sur `z`
(vue de dessus). On les contrôle par des invariants.

Le plus fort est le n° 5 : les deux vues intègrent le même `Δn` sur le même
volume, donc

$$\iint \mathrm{OPL}_{\text{côté}}\,\mathrm{d}x\,\mathrm{d}z \;=\; \iint \mathrm{OPL}_{\text{dessus}}\,\mathrm{d}x\,\mathrm{d}y$$

quelle que soit la physique. Un écart signale une erreur d'intégration, pas de modèle.


In [ ]:
for delay in (0.0, 500.0, 1901.0):
    ff.check_integration(res, delay_fs=delay, x_half_um=X_HALF_UM, **PROBE_KW)
    print()


### Bilan d'absorption

L'expérience donne **deux** transmittances pour le même plasma, avec des
longueurs de trajet très différentes : la vue de côté traverse une corde de
quelques µm, la vue de dessus toute la colonne. Le rapport des profondeurs
optiques est fixé par la géométrie et **ne dépend pas de σ** — c'est un test du
traitement, indépendant de tout modèle. Les valeurs absolues testent ensuite σ
et ρ_e.

Renseigner `T_MEASURED`, `CHORD_UM` et `COLUMN_UM` d'après la planche mesurée.


In [ ]:
T_MEASURED = 0.90     # T minimale lue sur la vue de COTE
CHORD_UM   = 5.0      # epaisseur du canal traverse par la sonde, vue de cote
COLUMN_UM  = 330.0    # longueur de la colonne, vue de dessus

print("=== modele historique ===")
ff.absorption_budget(res, 1901.0, x_half_um=X_HALF_UM, T_measured=T_MEASURED,
                     chord_um=CHORD_UM, column_um=COLUMN_UM, **PROBE_KW)

print("\n=== modele de permittivite (Martin et al. 1997) ===")
from permittivity import SIO2_MARTIN1997
ff.absorption_budget(res, 1901.0, x_half_um=X_HALF_UM, T_measured=T_MEASURED,
                     chord_um=CHORD_UM, column_um=COLUMN_UM,
                     material=SIO2_MARTIN1997, **PROBE_KW)

print()
ff.compare_probe_models(res, 1901.0, x_half_um=X_HALF_UM, **PROBE_KW)


## 7. Série de délais (run de référence)

In [ ]:
for delay in DELAYS_FS:
    fig, diag = ff.plot_opl_panel(
        res, delay_fs=float(delay),
        z_lim=Z_LIM_EXP, r_lim=R_LIM_EXP, x_half_um=X_HALF_UM,
        opl_clip_nm=15.0, t_lim=(0.75, 1.15),
        rho_max_cm3=RHO_MAX_CM3, validity_frac=0.1,
        title=f"simulation {ENERGY_INCIDENT_UJ:g} uJ, delay {delay/1000:+.3f} ps",
        save=str(FIG_DIR / f"delay_{delay:+05.0f}fs.png"),
        **PROBE_KW)
    print(f"{delay:+6.0f} fs :  OPL max = {np.abs(diag['opl_nm']).max():8.2f} nm"
          f"   T min = {diag['transmittance'].min():.3f}")

## 8. Décomposition par canal

In [ ]:
for channels in (("drude",), ("ste",), ("kerr",), ("drude", "ste", "kerr")):
    diag = ff.probe_opl_transmittance(res, 0.0, include=channels,
                                      x_half_um=X_HALF_UM, **PROBE_KW)
    print(f"  0 fs  {str(channels):32s} OPL max = {np.abs(diag['opl_nm']).max():8.2f} nm")

print("\nExperience : OPL ~15 nm, transmittance ~0.75")

## 9. Figures pour le rapport

Une figure par point du `main.tex`. Elles sont toutes calculées depuis `result.npz`,
aucun recalcul de propagation.

| fichier | section illustrée | ce qu'elle montre |
|---|---|---|
| `rep_clamping_equilibrium.png` | filamentation | $n_2 I$ contre $\rho_e/2\rho_c$ le long de $z$, et le $\Delta n$ net |
| `rep_selffocusing.png` | filamentation | rayon simulé contre propagation linéaire, $z_R$ et $L_c$ |
| `rep_avalanche_takeover.png` | avalanche | instant où $\beta I \rho_e$ dépasse la graine multiphotonique |
| `rep_trapping_sequence.png` | excitons / STE | $\rho_e \to \rho_{STE}$ jusqu'à 2 ps |
| `rep_index_channels.png` | Kerr / défocalisation plasma | profil radial des trois canaux de $\Delta n$ |
| `rep_abel_illustration.png` | interférométrie | $\Delta n(r)$ vrai contre sa projection Abel |
| `rep_energy_budget.png` | équation de propagation | photoionisation contre chauffage Drude |


In [ ]:
REPORT_DIAG = fr.export_report_figures(
    res, REP_DIR,
    wavelength_m=WAVELENGTH_M, n0=n0, n2=N2,
    w0_m=W0_M, energy_uJ=ENERGY_IN_GLASS_UJ, delta_t_s=DELTA_T_S,
    tau_c_s=TAU_C_S, Ui_eV=UI_EV, meff_rel=MEFF_REL, meff_drude_rel=MEFF_DRUDE_REL, begin_m=BEGIN_M, rho_max_cm3=RHO_MAX_CM3,
    tau_r_s=TAU_R_S, tau_ste_s=TAU_STE_S,
    lambda_probe_m=LAMBDA_PROBE_M, E_tr_eV=E_TR_EV,
    label=f"{ENERGY_INCIDENT_UJ:g} uJ",
    probe_kw=PROBE_KW)


### 9.1 Réglages individuels

Si une figure demande un autre plan ou un autre délai, la fonction correspondante
se rappelle seule ; `save=` écrase le PNG.


In [ ]:
# plan le plus intense par defaut ; z_target_um=... pour en choisir un autre
# delay_fs=None -> au passage de la pompe sur le plan trace
fr.plot_index_channels(res, delay_fs=None, lambda_probe_m=LAMBDA_PROBE_M,
                       E_tr_eV=E_TR_EV, n2=N2, tau_r_s=TAU_R_S,
                       tau_ste_s=TAU_STE_S, r_max_um=30.0,
                       label=f"{ENERGY_INCIDENT_UJ:g} uJ",
                       save=str(REP_DIR / "rep_index_channels.png"))

fr.plot_abel_illustration(res, delay_fs=None, x_half_um=40.0,
                          label=f"{ENERGY_INCIDENT_UJ:g} uJ",
                          save=str(REP_DIR / "rep_abel_illustration.png"),
                          **PROBE_KW)
print("regenerees")


### 9.2 Copie vers le projet LaTeX

Les `\includegraphics` du rapport cherchent les PNG à la racine de
`latex_project_text/`.


In [ ]:
import shutil

TEX_DIR = Path("../../latex_project_text").resolve()
if TEX_DIR.is_dir():
    for png in sorted(REP_DIR.glob("rep_*.png")):
        shutil.copy2(png, TEX_DIR / png.name)
        print("->", TEX_DIR / png.name)
else:
    print(f"{TEX_DIR} introuvable : copie manuelle des PNG de {REP_DIR}")
